# M29–M30 — Preuve d'idempotence du pipeline

Ce notebook remplace `scripts/prove_idempotence.ps1`. Meme demonstration, mais
executable pas a pas et lisible en seance.

## La question posee

> Que se passe-t-il si le pipeline tourne deux fois sur les memes donnees ?

Trois reponses possibles, une seule acceptable :

| Comportement | Diagnostic |
|---|---|
| Le nombre de lignes DOUBLE | `INSERT` sans garde : le contrat est rompu |
| Le nombre de lignes reste STABLE | Upsert correct |
| Rien n'est ecrit du tout | Idempotent, mais inutile |

Les deux derniers cas se ressemblent si l'on ne verifie que la stabilite. D'ou
**deux verdicts** a la fin de ce notebook, pas un seul.

## Ou tourne ce notebook

Il s'adapte a la base disponible :

* `INDUSENSE_DB_URL` definie -> cette base (PostgreSQL dans Compose) ;
* sinon -> SQLite temporaire, sans aucun prerequis.

La logique d'upsert est la meme dans les deux cas : `ON CONFLICT` a une syntaxe
identique en SQLite et PostgreSQL.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

from indusense.config import settings
from indusense.flows.predict_flow import predict_flow, taux_cible

# Base cible : celle de l'environnement, ou une SQLite jetable.
DB_URL = os.environ.get("INDUSENSE_DB_URL") or "sqlite:///artifacts/preuve_idempotence.db"
TABLE = settings.predictions_table
DATA_DIR = Path(os.environ.get("INDUSENSE_DATA_DIR") or settings.data_dir)

engine = create_engine(DB_URL)

# On n'affiche JAMAIS l'URL complete : elle contient le mot de passe.
print("Dialecte :", engine.dialect.name)
print("Base     :", DB_URL.rsplit("@", 1)[-1])
print("Donnees  :", DATA_DIR.resolve())
print("Table    :", TABLE)

## 1. Verifier les prerequis

Trois choses doivent etre en place avant de lancer quoi que ce soit. Verifier
d'abord evite de confondre « pipeline casse » et « environnement incomplet ».

In [ ]:
fichiers = [
    "capteurs_temperature.csv",
    "capteurs_pression.tsv",
    "releves_incidents.csv",
]

manquants = [f for f in fichiers if not (DATA_DIR / f).exists()]
if manquants:
    raise FileNotFoundError(
        f"Sources absentes de {DATA_DIR} : {manquants}. "
        "Definir INDUSENSE_DATA_DIR vers le jeu de donnees."
    )

modele = Path(settings.model_dir) / "rf.joblib"
if not modele.exists():
    raise FileNotFoundError(f"Modele absent : {modele}. Lancer l'entrainement d'abord.")

print("Sources et modele presents.")

## 2. Mesurer le jeu de donnees

Le taux de la colonne cible est le premier indicateur d'un jeu remplace ou
tronque. On le regarde AVANT de scorer : si les predictions deviennent absurdes
ensuite, on saura si l'entree y est pour quelque chose.

In [ ]:
from indusense.flows.predict_flow import ingest

dataset = ingest.fn(DATA_DIR, settings.incident_window_hours)

print(f"Lignes    : {len(dataset)}")
print(f"Machines  : {dataset['machine'].nunique()}  {sorted(dataset['machine'].unique())}")
print(f"Taux {settings.target_col} : {taux_cible(dataset):.2f} %")

### Valeur attendue

La fiche du jalon annonce **environ 4,78 %** sur le jeu complet du parcours.

Sur les donnees livrees avec le depot (4 machines), le taux vaut **10,42 %**.
Ce n'est pas une erreur : ce sont deux jeux differents.

La cellule suivante ne bloque donc PAS sur une egalite exacte. Elle verifie une
plage large, dont le seul role est d'attraper un jeu corrompu — taux a 0 % ou
au-dela de 25 %.

Pour figer la valeur exacte une fois le bon jeu monte, remplacer la plage par :

```python
assert abs(taux - 4.78) < 0.05
```

In [ ]:
taux = taux_cible(dataset)

if not (1.0 < taux < 25.0):
    raise ValueError(f"Taux hors plage plausible : {taux:.2f} %")

print(f"Taux dans la plage attendue : {taux:.2f} %")

## 3. Repartir d'une base propre

Sans cette purge, un run precedent fausserait le comptage : on ne saurait plus
si les lignes viennent de cette demonstration ou d'avant.

In [ ]:
with engine.begin() as cx:
    cx.execute(text(f"DROP TABLE IF EXISTS {TABLE}"))

print(f"Table {TABLE} supprimee (si elle existait).")

## 4. Premier passage

Le flow enchaine `ingest -> feature -> predict -> store`. Les logs Prefect
nomment chaque etape et sa duree.

In [ ]:
total_1 = predict_flow(data_dir=DATA_DIR, db_url=DB_URL, table=TABLE)

with engine.begin() as cx:
    count_1 = cx.execute(text(f"SELECT count(*) FROM {TABLE}")).scalar_one()

print(f"\nRUN 1 -> {count_1} lignes en base")

## 5. Second passage — strictement identique

Memes donnees, memes parametres. Tout ecart de comptage viendrait donc de
l'ecriture, pas de l'entree.

In [ ]:
total_2 = predict_flow(data_dir=DATA_DIR, db_url=DB_URL, table=TABLE)

with engine.begin() as cx:
    count_2 = cx.execute(text(f"SELECT count(*) FROM {TABLE}")).scalar_one()

print(f"\nRUN 2 -> {count_2} lignes en base")

## 6. Les deux verdicts

**Verdict 1 — idempotence.** Le compte ne doit pas bouger.

**Verdict 2 — population.** Le compte doit valoir ce qu'on attend. Un pipeline
qui n'ecrirait RIEN passerait le premier verdict sans rien prouver.

In [ ]:
# --- Verdict 1 : idempotence ---
if count_1 != count_2:
    raise AssertionError(f"Idempotence KO : {count_1} puis {count_2}")

print(f"VERDICT 1 OK — {count_1} lignes, stables sur deux passages.")

In [ ]:
# --- Verdict 2 : population scoree ---
# AJUSTER selon le jeu monte :
#   jeu complet du parcours       -> 15
#   donnees du depot (4 machines) -> 4
POPULATION_ATTENDUE = dataset["machine"].nunique()

if count_2 != POPULATION_ATTENDUE:
    raise AssertionError(
        f"Population scoree inattendue : {count_2} au lieu de {POPULATION_ATTENDUE}"
    )

print(f"VERDICT 2 OK — {count_2} lignes, une par machine.")

## 7. Lire le contenu

Une prediction par machine, sur son releve le plus recent.

In [ ]:
pd.read_sql(f"SELECT * FROM {TABLE} ORDER BY machine", engine)

## 8. Le piege — ce qui se passerait SANS upsert

Cette cellule reproduit un `INSERT` ordinaire, sans `ON CONFLICT`, dans une
table separee. Elle montre ce que le garde-fou evite.

C'est la demonstration qui compte en seance : **un pipeline sans upsert passe
tous les autres tests**. Il produit des predictions correctes, des logs
propres, un flow vert. Seul le comptage revele le probleme.

In [ ]:
TABLE_DEMO = f"{TABLE}_sans_upsert"

with engine.begin() as cx:
    cx.execute(text(f"DROP TABLE IF EXISTS {TABLE_DEMO}"))
    # Meme schema, mais SANS cle primaire : rien n'empeche les doublons.
    type_ts = "TEXT" if engine.dialect.name == "sqlite" else "TIMESTAMP"
    cx.execute(text(f'''
        CREATE TABLE {TABLE_DEMO} (
            machine       VARCHAR(32) NOT NULL,
            prediction_ts {type_ts}   NOT NULL,
            proba_panne   DOUBLE PRECISION NOT NULL
        )
    '''))

lignes = pd.read_sql(f"SELECT machine, prediction_ts, proba_panne FROM {TABLE}", engine)

comptes = []
for passage in (1, 2, 3):
    lignes.to_sql(TABLE_DEMO, engine, if_exists="append", index=False)
    with engine.begin() as cx:
        n = cx.execute(text(f"SELECT count(*) FROM {TABLE_DEMO}")).scalar_one()
    comptes.append(n)
    print(f"Passage {passage} sans upsert -> {n} lignes")

print(f"\nAvec upsert    : {count_1}, {count_2}  (stable)")
print(f"Sans upsert    : {', '.join(map(str, comptes))}  (croissance lineaire)")

In [ ]:
# Nettoyage de la table de demonstration.
with engine.begin() as cx:
    cx.execute(text(f"DROP TABLE IF EXISTS {TABLE_DEMO}"))

print("Table de demonstration supprimee.")

## A retenir

| Constat | Consequence |
|---|---|
| La cle est `(machine, prediction_ts)` | Deux horodatages differents = deux lignes, c'est voulu |
| `ON CONFLICT DO UPDATE` | Meme syntaxe en SQLite et PostgreSQL |
| SQLite stocke une CHAINE | D'ou `isoformat()` ; un `Timestamp` casserait la cle en silence |
| Sans upsert, le flow reste vert | Seul le COMPTAGE revele le probleme |

**Le point le plus important** : un pipeline non idempotent ne leve aucune
erreur. Il duplique, silencieusement, a chaque execution. C'est pourquoi la
preuve est un comptage, pas une relecture du code.

## Equivalent en ligne de commande

```powershell
uv run pytest tests/test_predict_flow.py -q      # memes garanties, en SQLite
.\scripts\prove_idempotence.ps1                 # dans Compose, sur PostgreSQL
```